# Następny krok: korekcja hubness w EEG → CLIP retrieval

Ten notebook powstał po analizie wyników w folderze `wyniki colab`, zwłaszcza po eksperymencie `retrieval_reranking_mole_unclip_embedding`.

Najważniejszy wynik poprzedniego kroku:

- raw nearest-neighbor po uśrednieniu powtórzeń: `top1 = 4.55%`, `top5 = 22.73%`, `category_top1 = 18.18%`, `SSIM = 0.206`,
- Stable UnCLIP EEG full: `SSIM = 0.173`,
- Stable UnCLIP oracle full: `SSIM = 0.246`,
- VAE ensemble dla `mole`: `SSIM = 0.286`.

Retrieval był lepszy niż bezpośredni UnCLIP z EEG, ale miał patologiczny objaw: wszystkie 44 obrazy testowe zostały przypisane tylko do 3 kandydatów (`apple_406570`, `airplane_128461`, `abstract_09_dot_grid`). To klasyczny problem hubness/collapse w retrievalu wysokowymiarowym. Ten notebook testuje metody korekcji tego zjawiska bez ponownego trenowania modelu.

## Hipoteza

Model EEG → CLIP ma słaby, ale realny sygnał. Problemem nie jest tylko sam poziom sygnału, ale to, że podobieństwo cosine faworyzuje kilku uniwersalnych kandydatów-hubów. Jeśli skorygujemy scoring kandydatów, możemy poprawić top-k i zgodność kategorii bez nowego treningu.

Testowane warianty:

1. `raw_cosine` — obecny punkt odniesienia.
2. `candidate_centered_beta_*` — odejmowanie średniej atrakcyjności kandydata.
3. `candidate_zscore` — standaryzacja wyniku per kandydat.
4. `csls_k5/k10` — Cross-domain Similarity Local Scaling, popularny trik anty-hubness.
5. `true_category_oracle_*` — górna granica: co by było, gdybyśmy znali kategorię obrazu i szukali tylko w niej.

In [ ]:
# Konfiguracja
from pathlib import Path

PARTICIPANT = 'mole'

DRIVE_DATA = Path('/content/drive/MyDrive/biai/data')
DRIVE_RESULTS = Path('/content/drive/MyDrive/biai/results')

EPOCHS_ZIP = DRIVE_DATA / 'biai_eeg_qc_0_0p8.zip'
ASSETS_ZIP = DRIVE_DATA / 'biai_unclip_assets.zip'

ROOT = Path('/content/biai_hubness_reranking')
MANIFEST_DIR = ROOT / f'reconstruction_manifests/participant_image_{PARTICIPANT}_no_abc'
EMBEDDING_DIR = ROOT / f'image_embeddings_unclip_participant_image_{PARTICIPANT}_no_abc'

RETRIEVAL_DIR = DRIVE_RESULTS / f'unclip_{PARTICIPANT}_retrieval'
UNCLIP_FULL_DIR = DRIVE_RESULTS / f'unclip_{PARTICIPANT}_generation_full'
RAW_RERANKING_DIR = DRIVE_RESULTS / f'retrieval_reranking_{PARTICIPANT}_unclip_embedding'
OUTPUT_DIR = DRIVE_RESULTS / f'hubness_corrected_reranking_{PARTICIPANT}_unclip_embedding'

GITHUB_BRANCH = 'codex-eeg-pipeline-qc'

VAE_BASELINE = {
    'model': 'VAE ensemble (mole)',
    'images': 44,
    'l1': 0.2355921593579379,
    'psnr': 11.59615940397436,
    'ssim': 0.2864757523956624,
}

print('ROOT =', ROOT)
print('RETRIEVAL_DIR =', RETRIEVAL_DIR)
print('RAW_RERANKING_DIR =', RAW_RERANKING_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# Drive + zależności
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'safetensors', 'scikit-learn'
], check=True)

print('Drive i zależności gotowe.')

In [ ]:
# Rozpakowanie danych i assets.
import shutil
import zipfile

def resolve_drive_file(path):
    path = Path(path)
    if path.is_file():
        return path
    drive_root = Path('/content/drive/MyDrive')
    matches = sorted(drive_root.rglob(path.name)) if drive_root.exists() else []
    if len(matches) == 1:
        print(f'Znalazłem {path.name} pod inną ścieżką:', matches[0])
        return matches[0]
    raise FileNotFoundError(f'Nie znalazłem {path}. Sprawdź MyDrive/biai/data/.')

if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

for source in map(resolve_drive_file, [EPOCHS_ZIP, ASSETS_ZIP]):
    print('Rozpakowuję:', source)
    with zipfile.ZipFile(source) as archive:
        archive.extractall(ROOT)

print('ROOT contents:', sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# Upewniamy się, że skrypty są dostępne. Starszy assets ZIP może ich nie mieć.
from urllib.request import urlretrieve

scripts_dir = ROOT / 'scripts'
scripts_dir.mkdir(exist_ok=True)
for script_name in ['train_eeg_image_retrieval.py', 'extract_unclip_image_embeddings.py']:
    script_path = scripts_dir / script_name
    if not script_path.is_file():
        raw_url = f'https://raw.githubusercontent.com/taf4you2/biai/{GITHUB_BRANCH}/scripts/{script_name}'
        print('Pobieram brakujący skrypt:', raw_url)
        urlretrieve(raw_url, script_path)

sys.path.insert(0, str(scripts_dir))
print('Skrypty gotowe:', sorted(p.name for p in scripts_dir.glob('*.py')))

In [ ]:
# Odtworzenie image embeddings, jeśli świeży runtime ich nie ma.
def run_logged(command, log_path, cwd=ROOT):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('RUN:', ' '.join(map(str, command)))
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError('Polecenie nie powiodło się: ' + ' '.join(map(str, command)))

if not (EMBEDDING_DIR / 'image_embeddings.npy').is_file():
    run_logged([
        sys.executable, '-u', 'scripts/extract_unclip_image_embeddings.py',
        '--manifest-dir', str(MANIFEST_DIR),
        '--project-root', str(ROOT),
        '--output-dir', str(EMBEDDING_DIR),
        '--batch-size', '8',
    ], DRIVE_RESULTS / f'hubness_reranking_{PARTICIPANT}_extract_embeddings.log')
else:
    print('Embeddingi już istnieją:', EMBEDDING_DIR)

In [ ]:
# Wczytanie modelu EEG→CLIP i wyliczenie embeddingów testowych po uśrednieniu powtórzeń.
import json
import math
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from torch.utils.data import DataLoader
from torchvision.transforms import functional as TF

from train_eeg_image_retrieval import EEGEmbeddingNet, EEGImageDataset, resolve_manifest_epoch_paths

checkpoint_path = RETRIEVAL_DIR / 'eeg_image_retrieval.pt'
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Brak checkpointu: {checkpoint_path}')

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
train_args = checkpoint['args']

test_frame = resolve_manifest_epoch_paths(
    pd.read_csv(MANIFEST_DIR / 'test.csv'),
    ROOT,
).reset_index(drop=True)

embedding_matrix = np.load(EMBEDDING_DIR / 'image_embeddings.npy').astype(np.float32)
embedding_index = pd.read_csv(EMBEDDING_DIR / 'image_embedding_index.csv')
image_to_embedding = dict(zip(embedding_index.image_id.astype(str), embedding_index.embedding_index.astype(int)))
category_lookup = dict(zip(embedding_index.image_id.astype(str), embedding_index.image_category.astype(str)))
image_lookup = dict(zip(embedding_index.image_id.astype(str), embedding_index.image_path.astype(str)))

dataset = EEGImageDataset(
    test_frame,
    image_to_embedding,
    image_to_embedding,
    embedding_matrix,
    checkpoint['mean'],
    checkpoint['std'],
    preload=True,
)
loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)

_, channels, samples = checkpoint['input_shape']
model = EEGEmbeddingNet(
    channels,
    samples,
    embedding_matrix.shape[1],
    f1=train_args['f1'],
    depth_multiplier=train_args['depth_multiplier'],
    f2=train_args['f2'],
    kernel_length=train_args['kernel_length'],
    dropout=train_args['dropout'],
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

predicted_batches = []
target_ids = []
with torch.inference_mode():
    for eeg, _, image_ids in loader:
        predicted_batches.append(model(eeg).cpu().numpy())
        target_ids.extend(image_ids)
predicted = np.concatenate(predicted_batches, axis=0).astype(np.float32)

sample_table = test_frame[['image_id', 'image_category']].copy()
sample_table['target_image_id'] = target_ids
sample_table['row_index'] = np.arange(len(sample_table))

candidate_table = test_frame[['image_id', 'image_category']].drop_duplicates().sort_values('image_id').reset_index(drop=True)
candidate_ids = candidate_table.image_id.astype(str).tolist()
candidate_categories = candidate_table.image_category.astype(str).tolist()
candidate_matrix = embedding_matrix[[image_to_embedding[image_id] for image_id in candidate_ids]]
candidate_matrix = candidate_matrix / np.maximum(np.linalg.norm(candidate_matrix, axis=1, keepdims=True), 1e-12)
candidate_position = {image_id: idx for idx, image_id in enumerate(candidate_ids)}

query_rows = []
query_embeddings = []
for target_id, group in sample_table.groupby('target_image_id', sort=True):
    indices = group.row_index.to_numpy()
    vector = predicted[indices].mean(axis=0)
    vector = vector / max(np.linalg.norm(vector), 1e-12)
    query_embeddings.append(vector)
    query_rows.append({
        'target_image_id': target_id,
        'target_category': category_lookup[target_id],
        'repetitions': len(indices),
    })

queries = pd.DataFrame(query_rows)
query_matrix = np.stack(query_embeddings).astype(np.float32)
raw_scores = query_matrix @ candidate_matrix.T

print('queries:', queries.shape, 'candidates:', len(candidate_ids), 'scores:', raw_scores.shape)

In [ ]:
# Metryki obrazowe i ewaluacja dowolnego scoringu.
def batch_ssim(predicted, target):
    c1, c2 = 0.01**2, 0.03**2
    mu_x = torch.nn.functional.avg_pool2d(predicted, 11, stride=1, padding=5)
    mu_y = torch.nn.functional.avg_pool2d(target, 11, stride=1, padding=5)
    sigma_x = torch.nn.functional.avg_pool2d(predicted.square(), 11, 1, 5) - mu_x.square()
    sigma_y = torch.nn.functional.avg_pool2d(target.square(), 11, 1, 5) - mu_y.square()
    sigma_xy = torch.nn.functional.avg_pool2d(predicted * target, 11, 1, 5) - mu_x * mu_y
    numerator = (2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)
    denominator = (mu_x.square() + mu_y.square() + c1) * (sigma_x + sigma_y + c2)
    return (numerator / denominator.clamp_min(1e-8)).mean(dim=(1, 2, 3))


def fit_image(path, size=256):
    with Image.open(path) as image:
        return ImageOps.fit(image.convert('RGB'), (size, size), method=Image.Resampling.LANCZOS)


def image_metrics(target_id, predicted_id, size=256):
    target = fit_image(image_lookup[target_id], size)
    predicted_image = fit_image(image_lookup[predicted_id], size)
    predicted_tensor = TF.to_tensor(predicted_image).unsqueeze(0)
    target_tensor = TF.to_tensor(target).unsqueeze(0)
    mse = (predicted_tensor - target_tensor).square().mean()
    return {
        'l1': float((predicted_tensor - target_tensor).abs().mean()),
        'mse': float(mse),
        'psnr': float(-10.0 * torch.log10(mse.clamp_min(1e-10))),
        'ssim': float(batch_ssim(predicted_tensor, target_tensor)[0]),
    }


def evaluate_score_matrix(method, scores, restrict_true_category=False):
    rows = []
    for query_index, query in queries.iterrows():
        row_scores = scores[query_index].copy()
        if restrict_true_category:
            allowed = np.array([category == query.target_category for category in candidate_categories])
            row_scores[~allowed] = -np.inf
        order = np.argsort(-row_scores)
        target_pos = candidate_position[query.target_image_id]
        rank = int(np.where(order == target_pos)[0][0] + 1)
        predicted_id = candidate_ids[int(order[0])]
        metrics = image_metrics(query.target_image_id, predicted_id)
        rows.append({
            'method': method,
            'target_image_id': query.target_image_id,
            'predicted_image_id': predicted_id,
            'target_category': query.target_category,
            'predicted_category': category_lookup[predicted_id],
            'rank': rank,
            'repetitions': int(query.repetitions),
            'l1': metrics['l1'],
            'mse': metrics['mse'],
            'psnr': metrics['psnr'],
            'ssim': metrics['ssim'],
        })
    frame = pd.DataFrame(rows)
    counts = frame.predicted_image_id.value_counts()
    candidate_count = len(candidate_ids)
    summary = {
        'method': method,
        'images': int(len(frame)),
        'top1': float((frame.rank <= 1).mean()),
        'top5': float((frame.rank <= min(5, candidate_count)).mean()),
        'top10': float((frame.rank <= min(10, candidate_count)).mean()),
        'category_top1': float((frame.target_category == frame.predicted_category).mean()),
        'median_rank': float(frame.rank.median()),
        'mean_rank': float(frame.rank.mean()),
        'l1': float(frame.l1.mean()),
        'mse': float(frame.mse.mean()),
        'psnr': float(frame.psnr.mean()),
        'ssim': float(frame.ssim.mean()),
        'distinct_predictions': int(counts.size),
        'top_prediction_share': float(counts.iloc[0] / len(frame)),
        'top3_prediction_share': float(counts.head(3).sum() / len(frame)),
        'restrict_true_category': bool(restrict_true_category),
    }
    return frame, summary

In [ ]:
# Budowa wariantów scoringu anty-hubness.
score_variants = {'raw_cosine': (raw_scores, False)}

candidate_mean = raw_scores.mean(axis=0, keepdims=True)
candidate_std = raw_scores.std(axis=0, keepdims=True).clip(min=1e-6)

for beta in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    score_variants[f'candidate_centered_beta_{beta:g}'] = (raw_scores - beta * candidate_mean, False)

score_variants['candidate_zscore'] = ((raw_scores - candidate_mean) / candidate_std, False)

def csls(scores, k):
    k = min(k, scores.shape[1], scores.shape[0])
    query_density = np.sort(scores, axis=1)[:, -k:].mean(axis=1, keepdims=True)
    candidate_density = np.sort(scores, axis=0)[-k:, :].mean(axis=0, keepdims=True)
    return 2 * scores - query_density - candidate_density

score_variants['csls_k5'] = (csls(raw_scores, 5), False)
score_variants['csls_k10'] = (csls(raw_scores, 10), False)
score_variants['true_category_oracle_raw'] = (raw_scores, True)
score_variants['true_category_oracle_centered_beta_1'] = (raw_scores - candidate_mean, True)

print('Metody:', list(score_variants))

In [ ]:
# Ewaluacja wszystkich metod i zapis wyników.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
method_frames = {}
summaries = []

for method, (scores, restrict_true_category) in score_variants.items():
    frame, summary = evaluate_score_matrix(method, scores, restrict_true_category=restrict_true_category)
    method_frames[method] = frame
    summaries.append(summary)
    frame.to_csv(OUTPUT_DIR / f'{method}_reconstructions.csv', index=False)

summary_df = pd.DataFrame(summaries).sort_values(['restrict_true_category', 'ssim', 'top5'], ascending=[True, False, False])
summary_df.to_csv(OUTPUT_DIR / 'method_comparison.csv', index=False)

with (OUTPUT_DIR / 'hubness_reranking_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({
        'participant': PARTICIPANT,
        'methods': summaries,
        'raw_reranking_dir': str(RAW_RERANKING_DIR),
        'retrieval_dir': str(RETRIEVAL_DIR),
    }, handle, indent=2)

display(summary_df.reset_index(drop=True))

In [ ]:
# Porównanie z VAE, UnCLIP i poprzednim raw rerankingiem.
unclip_summary = json.loads((UNCLIP_FULL_DIR / 'unclip_generation_summary.json').read_text(encoding='utf-8')) if (UNCLIP_FULL_DIR / 'unclip_generation_summary.json').is_file() else None
raw_previous = json.loads((RAW_RERANKING_DIR / 'reconstruction_summary.json').read_text(encoding='utf-8')) if (RAW_RERANKING_DIR / 'reconstruction_summary.json').is_file() else None

comparison_rows = [VAE_BASELINE]
if unclip_summary:
    comparison_rows.extend([
        {'model': 'Stable UnCLIP EEG full', 'images': unclip_summary['images'], **unclip_summary['eeg']},
        {'model': 'Stable UnCLIP oracle full', 'images': unclip_summary['images'], **unclip_summary['oracle']},
    ])
if raw_previous:
    comparison_rows.append({'model': 'raw nearest-neighbor previous', 'images': raw_previous['image_averaged']['samples'], **raw_previous['image_averaged_image_metrics']})

best_non_oracle = summary_df[~summary_df.restrict_true_category].sort_values('ssim', ascending=False).iloc[0]
best_oracle = summary_df[summary_df.restrict_true_category].sort_values('ssim', ascending=False).iloc[0]
comparison_rows.append({
    'model': f"best hubness-corrected: {best_non_oracle.method}",
    'images': int(best_non_oracle.images),
    'l1': float(best_non_oracle.l1),
    'mse': float(best_non_oracle.mse),
    'psnr': float(best_non_oracle.psnr),
    'ssim': float(best_non_oracle.ssim),
})
comparison_rows.append({
    'model': f"oracle category upper bound: {best_oracle.method}",
    'images': int(best_oracle.images),
    'l1': float(best_oracle.l1),
    'mse': float(best_oracle.mse),
    'psnr': float(best_oracle.psnr),
    'ssim': float(best_oracle.ssim),
})

comparison = pd.DataFrame(comparison_rows).sort_values('ssim', ascending=False).reset_index(drop=True)
comparison.to_csv(OUTPUT_DIR / 'comparison_against_unclip_vae.csv', index=False)
display(comparison)

In [ ]:
# Gridy dla raw, najlepszej metody anty-hubness i oracle kategorii.
def load_font(size):
    for path in [Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'), Path('C:/Windows/Fonts/arial.ttf')]:
        if path.exists():
            return ImageFont.truetype(str(path), size=size)
    return ImageFont.load_default()


def make_grid(frame, output_path, title, rows=8, image_size=190):
    selected = frame.sort_values('ssim', ascending=False).head(rows).reset_index(drop=True)
    gap = 14
    label_h = 64
    header_h = 44
    canvas = Image.new('RGB', (2 * image_size + gap + 30, header_h + rows * (image_size + label_h + gap) + 10), 'white')
    draw = ImageDraw.Draw(canvas)
    title_font = load_font(18)
    small_font = load_font(12)
    draw.text((15, 12), title, fill='black', font=title_font)
    y = header_h
    for _, row in selected.iterrows():
        target = fit_image(image_lookup[row.target_image_id], image_size)
        predicted_image = fit_image(image_lookup[row.predicted_image_id], image_size)
        canvas.paste(target, (15, y))
        canvas.paste(predicted_image, (15 + image_size + gap, y))
        color = '#2e7d32' if row.target_category == row.predicted_category else '#c62828'
        draw.rectangle((15 + image_size + gap, y, 15 + 2 * image_size + gap, y + image_size), outline=color, width=4)
        draw.text((15, y + image_size + 5), f"CEL {row.target_category}: {row.target_image_id[:28]}", fill='#333', font=small_font)
        draw.text((15 + image_size + gap, y + image_size + 5), f"PRED {row.predicted_category}: {row.predicted_image_id[:28]}", fill=color, font=small_font)
        draw.text((15 + image_size + gap, y + image_size + 25), f"rank={int(row.rank)} SSIM={row.ssim:.3f}", fill='#0d47a1', font=small_font)
        y += image_size + label_h + gap
    output_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output_path, quality=95)


grid_dir = OUTPUT_DIR / 'grids'
grid_methods = ['raw_cosine', str(best_non_oracle.method), str(best_oracle.method)]
for method in dict.fromkeys(grid_methods):
    make_grid(method_frames[method], grid_dir / f'{method}_best.jpg', f'{method} — najlepsze według SSIM')

for path in sorted(grid_dir.glob('*.jpg')):
    print(path)

In [ ]:
# Podgląd gridów.
from IPython.display import Image as IPImage, Markdown, display

for path in sorted((OUTPUT_DIR / 'grids').glob('*.jpg')):
    display(Markdown(f'### {path.name}'))
    display(IPImage(filename=str(path)))

In [ ]:
# Werdykt: czy anty-hubness rozwiązuje problem, czy tylko go diagnozuje?
from IPython.display import Markdown

raw = summary_df[summary_df.method == 'raw_cosine'].iloc[0]
best = best_non_oracle
oracle = best_oracle

lines = []
lines.append(f"Raw cosine: top5 `{raw.top5:.2%}`, category `{raw.category_top1:.2%}`, SSIM `{raw.ssim:.3f}`, distinct predictions `{int(raw.distinct_predictions)}`.")
lines.append(f"Najlepsza korekcja bez oracle: **{best.method}** — top5 `{best.top5:.2%}`, category `{best.category_top1:.2%}`, SSIM `{best.ssim:.3f}`, distinct predictions `{int(best.distinct_predictions)}`.")
lines.append(f"Oracle kategorii: **{oracle.method}** — top5 `{oracle.top5:.2%}`, category `{oracle.category_top1:.2%}`, SSIM `{oracle.ssim:.3f}`.")

if best.ssim > raw.ssim or best.distinct_predictions > raw.distinct_predictions:
    lines.append('Wniosek: korekcja hubness pomaga przynajmniej częściowo. Następny krok to użycie tej metody jako front-endu dla candidate-constrained generation.')
else:
    lines.append('Wniosek: prosta korekcja hubness nie wystarcza. Następny krok to poprawa samego dekodera EEG→embedding albo modelowanie kategorii jako osobnego etapu.')

if oracle.ssim > best.ssim + 0.03 or oracle.category_top1 > best.category_top1 + 0.10:
    lines.append('Duża luka do oracle kategorii oznacza, że warto osobno trenować/wykorzystać predyktor kategorii i dopiero potem rerankować wewnątrz kategorii.')

display(Markdown('\n\n'.join(lines)))